# 05 — Threshold Selection
## CRISP-DM Phase 4/5: Constrained Optimization -> tau = 0.1389

---

### CONCEPT
This notebook documents the derivation of the frozen decision threshold (tau = 0.1389) from development Out-of-Fold predictions.

### WHY
A probability model alone does not make screening decisions — a **decision threshold** is required. In screening contexts, missing a true case (false negative) carries greater public health cost than an unnecessary referral (false positive), so we prioritize high sensitivity.

### INPUT
- Development OOF predictions for GAM on EXPANDED_COMMON (N=3,232)
- Source: `predictions_phase4/oof_predictions.csv` and `reports_phase4_1/screening_operating_points.csv`
- Zero test data used in this analysis


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

plt.rcParams.update({
    'font.sans-serif': 'DejaVu Sans',
    'figure.autolayout': True,
    'figure.figsize': (12, 5)
})

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        def display(*args, **kwargs):
            for a in args:
                if hasattr(a, 'to_string'):
                    print(a.to_string())
                else:
                    print(a)

current = Path.cwd().resolve()
candidates = [current, current.parent, current.parent.parent]
PROJECT_ROOT = None
for c in candidates:
    if (c / "nhanes_feasibility_2021_2023").exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = current

NHANES_DIR = PROJECT_ROOT / "nhanes_feasibility_2021_2023"
PRED_DIR = NHANES_DIR / "predictions_phase4"
REPORTS_4_1 = NHANES_DIR / "reports_phase4_1"


## 1. The Constrained Optimization Problem & Mathematical Formulation

The decision threshold $\tau^*$ is derived as the solution to a **constrained optimization problem**:

$$\tau^* = \max \{ \tau \in [0, 1] \mid \text{Sensitivity}(\tau) \ge 0.90 \}$$

### Crucial Methodological Distinction:
- **The Constraint:** $\text{Sensitivity}(\tau) \ge 0.90$ (a pre-specified public health case-finding floor ensuring at least 90% of unrecognized dysglycemia cases are captured).
- **The Objective:** $\max \tau$ (maximizing the threshold, which monotonically increases **specificity**, thereby reducing false-positive referrals and minimizing unnecessary Stage-2 venipuncture laboratory burdens).
- **We are NOT maximizing sensitivity:** Pushing sensitivity arbitrarily higher (e.g. to 95% or 100%) severely degrades specificity and triggers unsustainable referral volumes. We maximize $\tau$ *subject to* the sensitivity floor.


In [2]:
# Load GAM development OOF predictions (N=3,232; zero test data)
oof_df = pd.read_csv(str(PRED_DIR / "oof_predictions.csv"))
gam_oof = oof_df[
    (oof_df["dataset_variant"] == "EXPANDED_COMMON") &
    (oof_df["model_family"] == "GAM") &
    (oof_df["model_configuration"] == "GAM_splines10_lam10.0")
].copy()

y_true = gam_oof["y_true"].values
y_prob = gam_oof["probability"].values

print("=== GAM Development Out-of-Fold (OOF) Predictions ===")
print(f"  Total Development Participants: N = {len(gam_oof):,}")
print(f"  True Dysglycemia Cases (Positive): {int(y_true.sum()):,} ({y_true.sum()/len(y_true)*100:.2f}%)")
print(f"  True Normal Cases (Negative):      {int((y_true == 0).sum()):,} ({(y_true==0).sum()/len(y_true)*100:.2f}%)")
print(f"  Predicted Probability Range:       [{y_prob.min():.4f}, {y_prob.max():.4f}]")


=== GAM Development Out-of-Fold (OOF) Predictions ===
  Total Development Participants: N = 3,232
  True Dysglycemia Cases (Positive): 747 (23.11%)
  True Normal Cases (Negative):      2,485 (76.89%)
  Predicted Probability Range:       [0.0118, 0.6810]


## 2. Historical Derivation: The 1,000-Point Search Grid

### Exact Origin of $\tau = 0.1389$
In the frozen research implementation (`scripts_phase4/run_phase4_1_audit.py`, lines 117–132), the operating threshold was derived via `find_highest_threshold_for_sensitivity_floor(y, p, 0.90)`:
- A uniform grid of 1,000 candidate thresholds on $[0.001, 0.999]$ was evaluated: $\tau_k = 0.001 + k \cdot \frac{0.998}{999}$.
- Grid point $k=138$ evaluated to $\tau = 0.13886186...$, achieving **$90.23\%$ Sensitivity** and **$42.25\%$ Specificity**.
- The subsequent grid step ($k=139$, $\tau = 0.139861...$) dropped sensitivity to **$89.83\%$**, failing the $\ge 90\%$ floor.
- In `screening_operating_points.csv`, this threshold was rounded to 4 decimal places as **`0.1389`**.


In [3]:
def compute_metrics_at_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    ref_pct = (tp + fp) / len(y_true) * 100.0
    tests_per_case = 1.0 / ppv if ppv > 0 else np.nan
    return {
        "threshold": threshold, "sensitivity": sens, "specificity": spec,
        "ppv": ppv, "npv": npv, "referred_pct": ref_pct, "tests_per_case": tests_per_case,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn
    }

# Reconstruct the historical 1,000-point uniform grid search from run_phase4_1_audit.py
thresholds = np.linspace(0.001, 0.999, 1000)
sweep_results = [compute_metrics_at_threshold(y_true, y_prob, t) for t in thresholds]
sweep_df = pd.DataFrame(sweep_results)

eligible = sweep_df[sweep_df["sensitivity"] >= 0.90].copy()
optimal = eligible.sort_values("threshold", ascending=False).iloc[0]

print("=== Constrained Optimization Grid Search (run_phase4_1_audit.py) ===")
print(f"Exact unrounded grid threshold: tau = {optimal['threshold']:.6f}")
print(f"Rounded frozen threshold (4 decimals): tau = {optimal['threshold']:.4f}")
print(f"  Sensitivity:    {optimal['sensitivity']*100:.2f}% (Constraint: >= 90.0%)")
print(f"  Specificity:    {optimal['specificity']*100:.2f}%")
print(f"  Referral Rate:  {optimal['referred_pct']:.2f}%")
print(f"  Tests/Case:     {optimal['tests_per_case']:.2f}")

assert round(optimal['threshold'], 4) == 0.1389, f"Mismatch with 0.1389: {optimal['threshold']}"


=== Constrained Optimization Grid Search (run_phase4_1_audit.py) ===
Exact unrounded grid threshold: tau = 0.138862
Rounded frozen threshold (4 decimals): tau = 0.1389
  Sensitivity:    90.23% (Constraint: >= 90.0%)
  Specificity:    42.25%
  Referral Rate:  65.25%
  Tests/Case:     3.13


## 3. Visualization: Sensitivity-Specificity Trade-Off


In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(sweep_df["threshold"], sweep_df["sensitivity"], color='#E74C3C', linewidth=2, label='Sensitivity')
ax1.plot(sweep_df["threshold"], sweep_df["specificity"], color='#3498DB', linewidth=2, label='Specificity')
ax1.axhline(y=0.90, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='90% Floor')
ax1.axvline(x=0.1389, color='#2ECC71', linestyle='-', linewidth=2, alpha=0.8, label='tau=0.1389')
ax1.set_xlabel('Threshold', fontsize=11)
ax1.set_ylabel('Metric Value', fontsize=11)
ax1.set_title('Sensitivity-Specificity vs Decision Threshold', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.set_xlim(0, 0.5)
ax1.grid(True, alpha=0.3)

ax2.plot(sweep_df["threshold"], sweep_df["referred_pct"], color='#9B59B6', linewidth=2)
ax2.axvline(x=0.1389, color='#2ECC71', linestyle='-', linewidth=2, alpha=0.8, label='tau=0.1389')
ax2.set_xlabel('Threshold', fontsize=11)
ax2.set_ylabel('Referral Rate (%)', fontsize=11)
ax2.set_title('Referral Rate vs Decision Threshold', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xlim(0, 0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Verified Operating Points from Phase 4.1


In [5]:
op_df = pd.read_csv(str(REPORTS_4_1 / "screening_operating_points.csv"))
gam_ops = op_df[
    (op_df["dataset_variant"] == "EXPANDED_COMMON") &
    (op_df["model_family"] == "GAM")
].copy()

print("=== GAM Operating Points (EXPANDED_COMMON, Development OOF) ===")
display(gam_ops[[
    "target_sensitivity_floor", "operating_threshold",
    "achieved_sensitivity", "achieved_specificity",
    "referred_percent", "hba1c_tests_per_case_detected"
]].reset_index(drop=True))

row_90 = gam_ops[gam_ops["target_sensitivity_floor"] == 0.9].iloc[0]
assert row_90["operating_threshold"] == 0.1389, f"Expected 0.1389, got {row_90['operating_threshold']}"
print(f"\ntau = 0.1389 confirmed from Phase 4.1 audit")


=== GAM Operating Points (EXPANDED_COMMON, Development OOF) ===
   target_sensitivity_floor  operating_threshold  achieved_sensitivity  achieved_specificity  referred_percent  hba1c_tests_per_case_detected
0                      0.80               0.1918                0.8005                0.5461             53.40                           2.89
1                      0.85               0.1668                0.8501                0.4865             59.13                           3.01
2                      0.90               0.1389                0.9023                0.4225             65.25                           3.13
3                      0.95               0.0909                0.9505                0.2885             76.67                           3.49

tau = 0.1389 confirmed from Phase 4.1 audit


## 5. Mandatory Audit: Threshold 0.1389 vs. Round Threshold 0.1400

### Why Retain 0.1389 Rather Than Simply Rounding to 0.1400?
A common question in thesis examination is: *"Why use 0.1389 instead of a cleaner round number like 0.14?"*
We answer this directly through computational evaluation of both thresholds on the actual development OOF predictions ($N=3,232$).


In [6]:
def evaluate_single_threshold(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    ref_rate = (tp + fp) / len(y_true) * 100.0
    return {
        "Threshold": threshold,
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Sensitivity": f"{sens*100:.2f}%",
        "Specificity": f"{spec*100:.2f}%",
        "PPV": f"{ppv*100:.2f}%",
        "NPV": f"{npv*100:.2f}%",
        "Referral_Rate": f"{ref_rate:.2f}%",
        "Meets_90pct_Floor": "PASS (>=90%)" if sens >= 0.90 else "FAIL (<90%)"
    }

res_1389 = evaluate_single_threshold(y_true, y_prob, 0.1389)
res_1400 = evaluate_single_threshold(y_true, y_prob, 0.1400)

comp_table = pd.DataFrame([res_1389, res_1400])
print("=== Threshold Comparison: 0.1389 vs 0.1400 on Development OOF ===")
display(comp_table)

# Observation-level discordance audit
discordant_mask = (y_prob >= 0.1389) != (y_prob >= 0.1400)
n_discordant = int(discordant_mask.sum())
disc_labels = y_true[discordant_mask]
n_missed_dys = int((disc_labels == 1).sum())
n_avoided_norm = int((disc_labels == 0).sum())

print(f"\nObservation-Level Discordance Audit:")
print(f"  Participants with predicted probability in [0.1389, 0.1400): {n_discordant}")
print(f"  Normal participants avoiding referral:                    {n_avoided_norm}")
print(f"  True dysglycemia cases MISSED (converted TP -> FN):        {n_missed_dys}")

assert n_missed_dys == 3, f"Expected 3 missed dysglycemia cases, got {n_missed_dys}"


=== Threshold Comparison: 0.1389 vs 0.1400 on Development OOF ===
   Threshold   TP    FP    TN  FN Sensitivity Specificity     PPV     NPV Referral_Rate Meets_90pct_Floor
0     0.1389  674  1435  1050  73      90.23%      42.25%  31.96%  93.50%        65.25%      PASS (>=90%)
1     0.1400  671  1424  1061  76      89.83%      42.70%  32.03%  93.32%        64.82%       FAIL (<90%)

Observation-Level Discordance Audit:
  Participants with predicted probability in [0.1389, 0.1400): 14
  Normal participants avoiding referral:                    11
  True dysglycemia cases MISSED (converted TP -> FN):        3


### INTERPRETATION — Exact Methodological Rationale for $\tau = 0.1389$
1. **Constraint Violation at 0.1400:**
   If the threshold is rounded up to $0.1400$, **sensitivity drops from $90.23\%$ to $89.83\%$**, which falls **below the pre-specified $\ge 90\%$ research operating floor**.
2. **Clinical Impact on Patients:**
   Exactly **14 participants** fall in the narrow probability interval $[0.1389, 0.1400)$.
   Among them, **3 participants have laboratory-confirmed dysglycemia**. Moving the threshold to $0.1400$ converts these 3 individuals from True Positives into False Negatives (missed cases increase from $73$ to $76$).
3. **Conclusion:**
   $\tau = 0.1389$ is retained not out of arbitrary precision, but because it represents the **exact mathematical feasibility boundary** of the constrained optimization problem on development data.

---

## 6. Specification Freeze

```text
THRESHOLD = 0.1389
STATUS = FROZEN
DERIVATION = Development OOF only (N=3,232)
POST_TEST_MODIFICATION = STRICTLY PROHIBITED
```


## AUDIT CHECK — Threshold Selection

| # | Verification Item | Status |
|:--|:------------------|:-------|
| 1 | Threshold derived from development OOF data only (zero test data) | PASS |
| 2 | Constrained optimization: max tau s.t. Sensitivity >= 90% | PASS |
| 3 | Historical 1,000-point uniform grid derivation documented | PASS |
| 4 | tau = 0.1389 matches Phase 4.1 audit report | PASS |
| 5 | Development sensitivity at tau=0.1389: 90.23% | PASS |
| 6 | Mandatory 0.1389 vs 0.1400 audit proves 0.1400 fails sensitivity floor (89.83%) | PASS |
| 7 | 3 missed dysglycemia cases at 0.1400 verified | PASS |
| 8 | Threshold freeze governance declared | PASS |
